# install

In [1]:
# Cài đặt hoặc nâng cấp vnstock
!pip install -U vnstock

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.7/278.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 2.5 MB/s eta 0:00:00


In [2]:
from vnstock import Quote
quote = Quote(symbol='ACB', source='KBS')


📋 Connecting Google Drive account
to save project settings.

Mounted at /content/drive


# Cell 1: Load dữ liệu

In [3]:
### Cell 1: Load dữ liệu (giữ nguyên)
from vnstock import Quote
import pandas as pd
import numpy as np

symbols = ["BID", "ACB", "VCB", "VNM", "MSN", "MWG", "HPG", "GAS", "SSI", "VRE"]

raw = {}
for sym in symbols:
    df = Quote(symbol=sym, source="KBS").history(start="2024-01-01", end="2025-12-31", interval="d")
    raw[sym] = df.set_index("time")[["close", "volume"]]

price_df = pd.DataFrame({sym: d["close"] for sym, d in raw.items()}).sort_index()
volume_df = pd.DataFrame({sym: d["volume"] for sym, d in raw.items()}).sort_index()

price_df.head()

,BID,ACB,VCB,VNM,MSN,MWG,HPG,GAS,SSI,VRE
time,,,,,,,,,,
2024-01-02 07:00:00,34.75,14.80,55.04,57.89,68.4,40.88,18.55,64.82,22.57,22.31
2024-01-03 07:00:00,35.40,15.13,55.70,58.49,68.9,41.60,18.78,65.17,22.88,22.45
2024-01-04 07:00:00,35.28,15.32,56.63,58.49,68.1,41.60,18.75,65.77,23.34,22.60
2024-01-05 07:00:00,35.97,15.41,56.82,58.32,67.9,42.23,18.78,66.19,23.72,22.55
2024-01-08 07:00:00,37.50,15.34,57.22,57.81,66.6,41.60,18.82,65.85,23.68,22.89


# Cell 2: Tính return


In [4]:
### Cell 2: Tính return
returns = price_df.pct_change().dropna()  # dropna vì hàng đầu tiên luôn NaN sau pct_change

returns.head()

,BID,ACB,VCB,VNM,MSN,MWG,HPG,GAS,SSI,VRE
time,,,,,,,,,,
2024-01-03 07:00:00,0.018705,0.022297,0.011991,0.010364,0.007310,0.017613,0.012399,0.005400,0.013735,0.006275
2024-01-04 07:00:00,-0.003390,0.012558,0.016697,0.000000,-0.011611,0.000000,-0.001597,0.009207,0.020105,0.006682
2024-01-05 07:00:00,0.019558,0.005875,0.003355,-0.002906,-0.002937,0.015144,0.001600,0.006386,0.016281,-0.002212
2024-01-08 07:00:00,0.042535,-0.004543,0.007040,-0.008745,-0.019146,-0.014918,0.002130,-0.005137,-0.001686,0.015078
2024-01-09 07:00:00,-0.007467,-0.013690,0.011534,-0.001557,-0.007508,-0.011538,-0.005845,-0.011693,-0.002956,-0.014854


# Cell 3: Tính covar matrix

In [5]:
### Cell 3: Tính covariance matrix
cov_matrix = returns.cov()

cov_matrix

,BID,ACB,VCB,VNM,MSN,MWG,HPG,GAS,SSI,VRE
BID,0.000240,0.000129,0.000117,0.000081,0.000134,0.000140,0.000128,0.000080,0.000161,0.000112
ACB,0.000129,0.000196,0.000095,0.000076,0.000121,0.000131,0.000122,0.000079,0.000161,0.000105
VCB,0.000117,0.000095,0.000160,0.000076,0.000100,0.000108,0.000100,0.000071,0.000123,0.000092
VNM,0.000081,0.000076,0.000076,0.000193,0.000108,0.000105,0.000094,0.000088,0.000112,0.000108
MSN,0.000134,0.000121,0.000100,0.000108,0.000346,0.000194,0.000170,0.000119,0.000205,0.000155
MWG,0.000140,0.000131,0.000108,0.000105,0.000194,0.000360,0.000169,0.000100,0.000227,0.000159
HPG,0.000128,0.000122,0.000100,0.000094,0.000170,0.000169,0.000262,0.000094,0.000205,0.000132
GAS,0.000080,0.000079,0.000071,0.000088,0.000119,0.000100,0.000094,0.000210,0.000105,0.000071
SSI,0.000161,0.000161,0.000123,0.000112,0.000205,0.000227,0.000205,0.000105,0.000400,0.000192
VRE,0.000112,0.000105,0.000092,0.000108,0.000155,0.000159,0.000132,0.000071,0.000192,0.000555


# Cell 4: Eigenvalues của covar matrix

In [6]:
### Cell 4: Tính eigenvalue của covariance matrix
# Covariance matrix luôn symmetric -> dùng eigh (nhanh và ổn định hơn eig cho ma trận đối xứng)
eigvals, eigvecs = np.linalg.eigh(cov_matrix.values)

eigvals_sorted = np.sort(eigvals)[::-1]  # sắp xếp giảm dần cho dễ đọc
pd.Series(eigvals_sorted, index=[f"λ{i+1}" for i in range(len(eigvals_sorted))])

,0
λ1,0.001480
λ2,0.000395
λ3,0.000195
λ4,0.000184
λ5,0.000153
λ6,0.000142
λ7,0.000110
λ8,0.000107
λ9,0.000084
λ10,0.000073


# Cell 5: Kiểm tra psd của covar matrix


In [7]:
### Cell 5: Kiểm tra Positive Semi-Definite (PSD)
# PSD: tất cả eigenvalue >= 0 (cho phép bằng 0)
TOL = 1e-8  # dung sai cho sai số làm tròn floating-point, tránh false negative do eigenvalue ~ -1e-16
is_psd = np.all(eigvals >= -TOL)

print(f"Min eigenvalue: {eigvals.min():.10f}")
print(f"Ma trận là PSD: {is_psd}")

Min eigenvalue: 0.0000730189
Ma trận là PSD: True


In [8]:
### Cell 6: Kiểm tra Positive Definite (PD) bằng Cholesky
# PD (mạnh hơn PSD): tất cả eigenvalue > 0 (không có 0) -> ma trận khả nghịch, Cholesky decompose được
# np.linalg.cholesky raise LinAlgError nếu ma trận không PD, dùng try/except để kiểm tra trực tiếp
try:
    L = np.linalg.cholesky(cov_matrix.values)
    is_pd = True
    print("Ma trận là PD -> Cholesky decomposition thành công.")
except np.linalg.LinAlgError:
    L = None
    is_pd = False
    print("Ma trận KHÔNG PD -> Cholesky thất bại (có eigenvalue <= 0, khả năng do đa cộng tuyến hoặc T < N).")

is_pd

Ma trận là PD -> Cholesky decomposition thành công.


True

# Cell 7+8: Dùng Cholesky, chuẩn bị cho Monte Carlo


In [9]:
### Cell 7: Dùng Cholesky để sinh random return phù hợp với covariance matrix
np.random.seed(42)
N_SIMS = 10_000  # số kịch bản Monte Carlo
N_ASSETS = len(symbols)

if L is None:
    raise ValueError("Không thể sinh random: covariance matrix không PD. Xem Cell 9 để kiểm tra nguyên nhân (T quá nhỏ).")

# Sinh standard normal độc lập (mean=0, std=1), rồi nhân với L để "tạo tương quan" đúng theo cov_matrix
z = np.random.standard_normal((N_SIMS, N_ASSETS))
correlated_shocks = z @ L.T  # shape (N_SIMS, N_ASSETS), Cov(correlated_shocks) ≈ cov_matrix

# Kiểm tra nhanh: covariance của sample sinh ra có khớp cov_matrix gốc không
sim_cov_check = pd.DataFrame(correlated_shocks, columns=symbols).cov()
print("Sai số trung bình giữa cov mô phỏng và cov gốc:", np.abs(sim_cov_check.values - cov_matrix.values).mean())

Sai số trung bình giữa cov mô phỏng và cov gốc: 8.30039785568955e-06


In [10]:
### Cell 8: Cộng thêm mean return để hoàn chỉnh chuỗi return mô phỏng cho Monte Carlo
mean_returns = returns.mean().values  # mean return hàng ngày, dùng làm drift cho simulation

simulated_returns = mean_returns + correlated_shocks  # shape (N_SIMS, N_ASSETS)
simulated_returns_df = pd.DataFrame(simulated_returns, columns=symbols)

simulated_returns_df.describe().T[["mean", "std"]]  # so sánh nhanh với returns.describe() gốc để sanity-check

,mean,std
BID,0.000363,0.015847
ACB,0.000772,0.014149
VCB,0.000082,0.012836
VNM,0.000034,0.013875
MSN,0.000568,0.018898
MWG,0.001769,0.019338
HPG,0.000792,0.016565
GAS,0.000351,0.014541
SSI,0.000895,0.020380
VRE,0.000940,0.023692


# Cell 9: Kiểm tra số quan sát

In [11]:
### Cell 9: Kiểm tra số quan sát T cho mỗi mã, đảm bảo điều kiện PD (T >= N)
# Điều kiện cần (không đủ) để cov_matrix full-rank/PD: T (số quan sát) phải >= N (số tài sản)
# Nếu T < N, cov_matrix chắc chắn singular (rank-deficient) -> không thể PD, chỉ có thể PSD nhất

obs_count = returns.count()  # đếm số quan sát non-NaN cho từng mã
min_T = obs_count.min()

print(f"Số tài sản N = {N_ASSETS}")
print(f"Số quan sát T nhỏ nhất trong các mã: {min_T} (mã: {obs_count.idxmin()})")
print(f"Điều kiện cần T >= N: {'THỎA MÃN' if min_T >= N_ASSETS else 'VI PHẠM'}")

obs_count.sort_values()

Số tài sản N = 10
Số quan sát T nhỏ nhất trong các mã: 498 (mã: BID)
Điều kiện cần T >= N: THỎA MÃN


,0
BID,498
ACB,498
VCB,498
VNM,498
MSN,498
MWG,498
HPG,498
GAS,498
SSI,498
VRE,498


Các mã đều đủ quan sát, đủ bằng chứng chứng minh ma trận covar có PD, và có thể sử dụng Cholesky kết hợp với Monte Carlo để mô phỏng danh mục

# Cell 10: Lấy lại dữ liệu để test

In [12]:
### Cell 10: Lấy 7 dòng return (T=7 < N=10) và tính lại covariance matrix
T_SMALL = 7  # cố ý chọn T < N=10 để minh họa trường hợp vi phạm điều kiện cần cho PD

returns_small = returns.iloc[:T_SMALL]
cov_matrix_small = returns_small.cov()

print(f"T (số quan sát) = {T_SMALL}, N (số tài sản) = {len(symbols)}")
print(f"T < N: {T_SMALL < len(symbols)}  ->  cov_matrix chắc chắn rank-deficient (rank tối đa = T-1 = {T_SMALL - 1})")

cov_matrix_small

T (số quan sát) = 7, N (số tài sản) = 10
T < N: True  ->  cov_matrix chắc chắn rank-deficient (rank tối đa = T-1 = 6)


,BID,ACB,VCB,VNM,MSN,MWG,HPG,GAS,SSI,VRE
BID,0.000485,0.000053,0.000047,-0.000088,-0.000030,-0.000054,-0.000013,-0.000047,-0.000067,0.000073
ACB,0.000053,0.000141,0.000031,0.000036,0.000060,0.000100,0.000034,0.000063,0.000070,0.000042
VCB,0.000047,0.000031,0.000057,-0.000015,0.000004,-0.000032,-0.000032,-0.000017,-0.000015,-0.000015
VNM,-0.000088,0.000036,-0.000015,0.000050,0.000036,0.000068,0.000042,0.000032,0.000043,0.000006
MSN,-0.000030,0.000060,0.000004,0.000036,0.000077,0.000073,0.000015,0.000018,0.000010,-0.000024
MWG,-0.000054,0.000100,-0.000032,0.000068,0.000073,0.000171,0.000072,0.000081,0.000111,0.000013
HPG,-0.000013,0.000034,-0.000032,0.000042,0.000015,0.000072,0.000063,0.000036,0.000051,0.000040
GAS,-0.000047,0.000063,-0.000017,0.000032,0.000018,0.000081,0.000036,0.000064,0.000076,0.000037
SSI,-0.000067,0.000070,-0.000015,0.000043,0.000010,0.000111,0.000051,0.000076,0.000121,0.000028
VRE,0.000073,0.000042,-0.000015,0.000006,-0.000024,0.000013,0.000040,0.000037,0.000028,0.000092


# Cell 11: Bằng chứng Cholesky thất bại với ma trận PSD

In [13]:
### Cell 11: Chứng minh eigenvalue âm và Cholesky thất bại
eigvals_small, _ = np.linalg.eigh(cov_matrix_small.values)
eigvals_small_sorted = np.sort(eigvals_small)[::-1]

print("Eigenvalues (giảm dần):")
print(pd.Series(eigvals_small_sorted, index=[f"λ{i+1}" for i in range(len(eigvals_small_sorted))]).round(8))
print(f"\nSố eigenvalue âm: {(eigvals_small < -1e-8).sum()} / {len(eigvals_small)}")
print(f"Min eigenvalue: {eigvals_small.min():.10f}  ->  {'KHÔNG PSD (có giá trị âm)' if eigvals_small.min() < -1e-8 else 'PSD'}")

try:
    np.linalg.cholesky(cov_matrix_small.values)
    print("\nCholesky: THÀNH CÔNG")
except np.linalg.LinAlgError as e:
    print(f"\nCholesky: THẤT BẠI -> {e}")

Eigenvalues (giảm dần):
λ1     0.000584
λ2     0.000428
λ3     0.000141
λ4     0.000085
λ5     0.000057
λ6     0.000025
λ7     0.000000
λ8     0.000000
λ9    -0.000000
λ10   -0.000000
dtype: float64

Số eigenvalue âm: 0 / 10
Min eigenvalue: -0.0000000000  ->  PSD

Cholesky: THẤT BẠI -> Matrix is not positive definite


# Cell 12: Khắc phục bằng Shrinkage

In [14]:
### Cell 12: Shrinkage đơn giản để khôi phục PD

def shrink_cov(cov: pd.DataFrame, alpha: float = 0.3) -> pd.DataFrame:
    """
    Linear shrinkage đơn giản (kiểu Ledoit-Wolf rút gọn): kéo sample covariance về gần
    một ma trận target "an toàn" (diagonal, phương sai trung bình) theo tỷ trọng alpha.

    shrunk = (1 - alpha) * sample_cov + alpha * target
    target = ma trận đường chéo với phương sai trung bình của toàn bộ tài sản (loại bỏ hết off-diagonal noise)

    alpha càng lớn -> càng "an toàn" (gần identity-scaled) nhưng càng mất thông tin tương quan gốc.
    alpha=0 -> giữ nguyên sample cov (có thể không PD); alpha=1 -> chỉ còn target (luôn PD nếu variance>0).
    """
    n = cov.shape[0]
    avg_var = np.trace(cov.values) / n
    target = np.eye(n) * avg_var
    shrunk = (1 - alpha) * cov.values + alpha * target
    return pd.DataFrame(shrunk, index=cov.index, columns=cov.columns)

cov_shrunk = shrink_cov(cov_matrix_small, alpha=0.3)

eigvals_shrunk, _ = np.linalg.eigh(cov_shrunk.values)
print(f"Min eigenvalue sau shrinkage (alpha=0.3): {eigvals_shrunk.min():.8f}")

try:
    L_shrunk = np.linalg.cholesky(cov_shrunk.values)
    print("Cholesky sau shrinkage: THÀNH CÔNG -> ma trận đã PD")
except np.linalg.LinAlgError as e:
    print(f"Cholesky sau shrinkage: vẫn THẤT BẠI -> {e} (thử tăng alpha)")

Min eigenvalue sau shrinkage (alpha=0.3): 0.00003962
Cholesky sau shrinkage: THÀNH CÔNG -> ma trận đã PD
